# 그래프 합성곱 QSAR 모델 기반 분자의 원자 기여도(Atomic Contribution) 계산

앞선 튜토리얼에서 우리는 모델 해석 가능성(interpretability), 즉 모델이 왜 그런 결과를 냈는지 이해하는 개념을 소개했습니다. 이번 튜토리얼에서는 분자에 작동하는 모델을 해석하는 데 유용한 도구인 원자 기여도(atomic contribution)에 대해 배웁니다.

아이디어는 간단합니다. 분자에서 원자(atom) 하나를 제거하고, 모델의 예측이 어떻게 변하는지 보는 것입니다. 어떤 원자의 "원자 기여도"는 분자 전체의 활성(activity)과, 그 원자를 제거한 뒤 남은 조각(fragment)의 활성 간 차이로 정의됩니다. 이는 해당 원자가 예측에 얼마나 영향을 주는지를 나타내는 척도입니다.

기여도(contribution)는 문헌에서 "기여(attribution)", "채색(coloration)" 등으로도 불립니다. 이것은 모델 해석 기법 [1]으로, QSAR 분야의 유사도 지도(Similarity map) [2]나, 다른 분야(이미지 분류 등)의 가림(occlusion) 기법과 유사합니다. 본 구현은 [4]에서 사용되었습니다.

Mariia Matveieva, Pavel Polishchuk. Institute of Molecular and Translational Medicine, Palacky University, Olomouc, Czech Republic.

<img src="https://github.com/deepchem/deepchem/blob/master/examples/tutorials/assets/atomic_contributions_tutorial_data/index.png?raw=1">

## Colab에서 실행하기

이 노트북은 Google Colab에서 실행하는 것을 권장합니다. 아래 Colab 배지를 클릭하면 바로 열립니다. (Kaggle에서도 열 수 있습니다.)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/isg-yhlee93/lecture/blob/main/Molecular%20Machine%20Learning/5_Atomic_Contributions_for_Molecules.ipynb)
[![Open In Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://www.kaggle.com/kernels/welcome?src=https://github.com/isg-yhlee93/lecture/blob/main/Molecular%20Machine%20Learning/5_Atomic_Contributions_for_Molecules.ipynb)

## 설치(Setup)

아래 셀을 실행하여 DeepChem을 설치합니다.


In [ ]:
!pip install -qq --pre deepchem
import deepchem

# 불필요한 경고·로그 메시지 끄기 (화면을 깔끔하게 보기 위함)
import warnings
warnings.filterwarnings('ignore')   # 파이썬 경고 숨기기

from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')      # RDKit 로그 끄기

deepchem.__version__


## 혈액-뇌 장벽(BBB) 투과성에 대한 분류 QSAR 모델

BBB 투과성(BBB permeability)은 화합물이 중추신경계(central nervous system)로 들어갈 수 있는 능력을 말합니다. 여기서는 운반체(carrier) 없이 확산(diffusion)으로 이동하는 비교적 작은 화합물들의 데이터셋을 사용합니다. 이 성질은 log10(뇌 내 농도 / 혈액 내 농도)로 정의됩니다. 값이 양수(및 0)인 화합물은 활성(active), 그 외는 비활성(inactive)으로 레이블됩니다. 모델링 이후, 우리는 확산에 유리한 원자와 불리한 원자를 식별할 것입니다.

먼저 데이터셋을 만들어 봅시다. 분자들은 SDF 파일에 저장되어 있습니다.


In [ ]:
import os
import pandas as pd
import deepchem as dc
import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem import Draw, PyMol, rdFMCS
from rdkit.Chem.Draw import IPythonConsole
from rdkit import rdBase
from deepchem import metrics
from IPython.display import Image, display, SVG
from rdkit.Chem.Draw import SimilarityMaps
import torch

current_dir = os.path.dirname(os.path.realpath('__file__'))
# BBB 투과성 데이터(SDF)를 내려받습니다.
dc.utils.download_url(
    'https://raw.githubusercontent.com/deepchem/deepchem/master/examples/tutorials/assets/atomic_contributions_tutorial_data/logBB.sdf',
    current_dir,
    'logBB.sdf'
)
DATASET_FILE =os.path.join(current_dir, 'logBB.sdf')
# 나중에 필요하므로 RDKit 분자 객체도 만들어 둡니다.
mols = [m for m in Chem.SDMolSupplier(DATASET_FILE) if m is not None ]
loader = dc.data.SDFLoader(tasks=["logBB_class"], 
                           featurizer=dc.feat.ConvMolFeaturizer(),
                           sanitize=True)
dataset = loader.create_dataset(DATASET_FILE, shard_size=2000)


이제 GraphConvModel을 만들고 학습시켜 봅시다.


In [ ]:
# 재현성을 위해 난수 시드를 고정합니다.
np.random.seed(2020)
torch.manual_seed(2020)


In [ ]:
number_input_features = (dataset.X[0].get_atom_features().shape[1], 64) # 64는 두 번째 층의 입력 특징 수
m = dc.models.torch_models.GraphConvModel(1, mode="classification", number_input_features=number_input_features, batch_normalize=False, batch_size=100, device='cpu')
m.fit(dataset, nb_epoch=10)


테스트 세트를 불러와 모델이 얼마나 잘 동작하는지 확인해 봅시다.


In [ ]:
current_dir = os.path.dirname(os.path.realpath('__file__'))
dc.utils.download_url(
    'https://raw.githubusercontent.com/deepchem/deepchem/master/examples/tutorials/assets/atomic_contributions_tutorial_data/logBB_test_.sdf',
    current_dir,
    'logBB_test_.sdf'
)
TEST_DATASET_FILE = os.path.join(current_dir, 'logBB_test_.sdf')
loader = dc.data.SDFLoader(tasks=["p_np"], sanitize=True,
                           featurizer=dc.feat.ConvMolFeaturizer())
test_dataset = loader.create_dataset(TEST_DATASET_FILE, shard_size=2000)
pred =  m.predict(test_dataset)
pred = np.argmax(np.squeeze(pred),axis=1)
# 균형 정확도(balanced accuracy)로 성능을 평가합니다.
ba = metrics.balanced_accuracy_score(y_true=test_dataset.y, y_pred=pred)
print(ba)


균형 정확도가 충분히 높습니다. 이제 모델 해석으로 넘어가, 예측에 대한 개별 원자의 기여도를 추정해 봅시다.


## 조각(fragment) 데이터셋

이제 학습 세트를 기반으로 조각(fragment) 데이터셋을 준비해 봅시다. (관심 있는 다른 미지의 데이터셋을 사용해도 됩니다.) 이 조각들은 개별 원자의 기여도를 평가하는 데 사용됩니다.

각 분자에 대해 ConvMol 객체들의 목록을 생성합니다. `per_atom_fragmentation=True`를 지정하면, 모든 무거운 원자(heavy atom)를 순회하면서 각 원자를 하나씩 제거한 버전의 분자를 특징화하게 됩니다.


In [ ]:
loader = dc.data.SDFLoader(tasks=[],# 작업(task)이 필요 없습니다 (오히려 작업을 넘기면 데이터 형태 불일치가 생길 수 있음)
                        featurizer=dc.feat.ConvMolFeaturizer(per_atom_fragmentation=True),
                        sanitize=True) 
frag_dataset = loader.create_dataset(DATASET_FILE, shard_size=5000)


이 데이터셋은 여전히 원래 학습 세트와 같은 수의 샘플을 갖지만, 이제 각 샘플은 하나의 ConvMol이 아니라 ConvMol 객체들의 목록(조각마다 하나씩)으로 표현됩니다.

중요: 조각의 순서는 입력 형식에 따라 달라집니다. SDF라면 조각 순서는 해당 mol 블록의 원자 순서와 같습니다. SMILES(즉 분자가 SMILES로 표현된 csv)라면, 순서는 RDKit의 CanonicalRankAtoms로 결정됩니다.


In [ ]:
print(frag_dataset.X.shape)


우리는 각 조각을 별개의 샘플로 다루고 싶습니다. FlatteningTransformer를 사용해 조각 목록을 평탄화(flatten)할 수 있습니다.


In [ ]:
# 조각 목록을 평탄화하여 각 조각을 개별 샘플로 만듭니다.
tr = dc.trans.FlatteningTransformer(frag_dataset)
frag_dataset = tr.transform(frag_dataset)
print(frag_dataset.X.shape)


## 활성에 대한 원자 기여도 예측하기

이제 분자와 그 조각들에 대해 활성을 예측합니다. 그런 다음 각 조각에 대해 활성 차이, 즉 원자 하나를 제거했을 때의 활성 변화를 구합니다.

참고: 여기 분류(classification) 맥락에서는 모델의 확률(probability) 출력을 활성으로 사용합니다. 따라서 기여도는 확률 차이가 됩니다. 즉 "해당 원자가 분자가 활성일 확률을 얼마나 높이거나 낮추는가"를 의미합니다.


In [ ]:
# 분자 전체
pred = np.squeeze(m.predict(dataset))[:, 1] # 클래스 1(활성)일 확률
pred = pd.DataFrame(pred, index=dataset.ids, columns=["Molecule"])  # 편의를 위해 데이터프레임으로 변환

# 조각들
pred_frags = np.squeeze(m.predict(frag_dataset))[:, 1]
pred_frags = pd.DataFrame(pred_frags, index=frag_dataset.ids, columns=["Fragment"])


차이를 구해 원자 기여도를 찾습니다.


In [ ]:
# 분자 이름을 기준으로 두 데이터프레임을 병합합니다.
df = pd.merge(pred_frags, pred, right_index=True, left_index=True)
# 기여도 = 분자 전체 확률 - 조각 확률
df['Contrib'] = df["Molecule"] - df["Fragment"]


In [ ]:
df


RDKit의 SimilarityMaps 기능을 사용해 결과를 시각화할 수 있습니다. 각 원자는 활성에 어떤 영향을 주는지에 따라 색이 칠해집니다.


In [ ]:
def vis_contribs(mols, df, smi_or_sdf = "sdf"): 
    # 데이터셋을 만들 때 사용한 파일의 입력 형식이 원자 순서를 결정하므로,
    # 올바른 매핑을 위해 이를 고려합니다!
    maps = []
    for mol in mols:
        wt = {}
        if smi_or_sdf == "smi":
            for n, atom in enumerate(Chem.rdmolfiles.CanonicalRankAtoms(mol)):
                wt[atom] = df.loc[mol.GetProp("_Name"), "Contrib"][n]
        if smi_or_sdf == "sdf":
            for n, atom in enumerate(range(mol.GetNumHeavyAtoms())):
                wt[atom] = df.loc[Chem.MolToSmiles(mol), "Contrib"][n]

        # 그리기 객체를 생성하고 유사도 지도를 SVG로 그립니다.
        # (최신 RDKit은 GetSimilarityMapFromWeights에 draw2d 객체를 필수로 요구합니다)
        drawObj = Draw.MolDraw2DSVG(700, 700)
        SimilarityMaps.GetSimilarityMapFromWeights(mol, wt, drawObj, contourLines=10)
        drawObj.FinishDrawing()
        map = drawObj.GetDrawingText()
        maps.append(map)
    return maps


몇 가지 그림을 살펴봅시다:


In [ ]:
np.random.seed(2000)
maps = vis_contribs(np.random.choice(np.array(mols),10), df)
for map in maps:
      display(SVG(map))


방향족(aromatic) 또는 지방족(aliphatic) 구조는 혈액-뇌 장벽 투과성에 긍정적인 영향을 주는 반면, 극성(polar)이거나 전하를 띤 헤테로원자(heteroatom)는 부정적인 영향을 준다는 것을 볼 수 있습니다. 이는 대체로 문헌 데이터와 일치합니다.


## 회귀(regression) 과제

위 예제는 분류 모델을 사용했습니다. 같은 기법을 회귀 모델에도 적용할 수 있습니다. 수생 독성(aquatic toxicity, 수생 생물 T. pyriformis에 대한 독성)이라는 회귀 과제를 살펴봅시다.

독성은 log10(IGC50)(군집 성장을 50% 억제하는 농도)으로 정의됩니다. 원자 기여도를 통해 T. pyriformis에 대한 독성단(toxicophore)을 식별할 것입니다.

위의 모든 단계는 동일합니다. 데이터 불러오기, 특징화, 모델 구축, 조각 데이터셋 생성, 기여도 계산, 시각화.

참고: 이번에는 회귀이므로 기여도는 확률이 아니라 활성 단위(activity unit)로 표현됩니다.


In [ ]:
current_dir = os.path.dirname(os.path.realpath('__file__'))
dc.utils.download_url(
    'https://raw.githubusercontent.com/deepchem/deepchem/master/examples/tutorials/assets/atomic_contributions_tutorial_data/Tetrahymena_pyriformis_Work_set_OCHEM.sdf',
    current_dir,
    'Tetrahymena_pyriformis_Work_set_OCHEM.sdf'
)
DATASET_FILE =os.path.join(current_dir, 'Tetrahymena_pyriformis_Work_set_OCHEM.sdf')

# 나중에 필요하므로 RDKit 분자 객체를 만들어 둡니다.
mols = [m for m in Chem.SDMolSupplier(DATASET_FILE) if m is not None ]
loader = dc.data.SDFLoader(tasks=["IGC50"], 
                           featurizer=dc.feat.ConvMolFeaturizer(), sanitize=True)
dataset = loader.create_dataset(DATASET_FILE, shard_size=5000)


모델을 만들고 학습시킵니다.


In [ ]:
np.random.seed(2020)
torch.manual_seed(2020)
number_input_features = (dataset.X[0].get_atom_features().shape[1], 64) # 64는 두 번째 층의 입력 특징 수
m = dc.models.torch_models.GraphConvModel(1, mode="regression", number_input_features=number_input_features, batch_normalize=False, device='cpu')
m.fit(dataset, nb_epoch=80)


테스트 데이터셋을 불러와 모델 성능을 확인합니다.


In [ ]:
current_dir = os.path.dirname(os.path.realpath('__file__'))
dc.utils.download_url(
    'https://raw.githubusercontent.com/deepchem/deepchem/master/examples/tutorials/assets/atomic_contributions_tutorial_data/Tetrahymena_pyriformis_Test_set_OCHEM.sdf',
    current_dir,
    'Tetrahymena_pyriformis_Test_set_OCHEM.sdf'
)




TEST_DATASET_FILE = os.path.join(current_dir, 'Tetrahymena_pyriformis_Test_set_OCHEM.sdf')
loader = dc.data.SDFLoader(tasks=["IGC50"], sanitize= True,
                           featurizer=dc.feat.ConvMolFeaturizer())
test_dataset = loader.create_dataset(TEST_DATASET_FILE, shard_size=2000)
pred = m.predict(test_dataset)
# 평균제곱오차(MSE)와 결정계수(R^2)로 평가합니다.
mse = metrics.mean_squared_error(y_true=test_dataset.y, y_pred=pred)
r2 = metrics.r2_score(y_true=test_dataset.y, y_pred=pred)
print(mse)
print(r2)


학습 세트를 다시 불러오되, 이번에는 `per_atom_fragmentation=True`로 설정합니다.


In [ ]:
loader = dc.data.SDFLoader(tasks=[], # 작업(task)이 필요 없습니다
                           sanitize=True,
                           featurizer=dc.feat.ConvMolFeaturizer(per_atom_fragmentation=True)) 
frag_dataset = loader.create_dataset(DATASET_FILE, shard_size=5000)
tr = dc.trans.FlatteningTransformer(frag_dataset) # 데이터셋을 평탄화하고 각 조각에 id를 부여합니다
frag_dataset = tr.transform(frag_dataset)


활성 차이를 계산합니다.


In [ ]:
# 분자 전체
pred = m.predict(dataset)
pred = pd.DataFrame(pred, index=dataset.ids, columns=["Molecule"])  # 편의를 위해 데이터프레임으로 변환

# 조각들
pred_frags = m.predict(frag_dataset)
pred_frags = pd.DataFrame(pred_frags, index=frag_dataset.ids, columns=["Fragment"])  # 편의를 위해 데이터프레임으로 변환

# 분자 이름을 기준으로 두 데이터프레임을 병합합니다.
df = pd.merge(pred_frags, pred, right_index=True, left_index=True)
# 기여도 = 분자 전체 활성 - 조각 활성
df['Contrib'] = df["Molecule"] - df["Fragment"]


활성이 중간 정도인(지나치게 활성/비활성이 아닌) 분자 몇 개를 골라 원자 기여도를 시각화해 봅시다.


In [ ]:
maps = vis_contribs([mol for mol in mols if float(mol.GetProp("IGC50"))>3 and float(mol.GetProp("IGC50"))<4][:10], df)


알려진 독성단(toxicophore)이 초록색으로 표시된 것을 볼 수 있습니다. 즉 니트로-방향족(nitro-aromatics), 할로-방향족(halo-aromatics), 긴 알킬 사슬(long alkyl chain), 알데하이드(aldehyde) 등입니다. 반면 카복실기(carboxylic group), 알코올(alcohol), 아미노(amino)기는 독성을 낮추는데, 이는 문헌 [3]과 일치합니다.


# 부록(Appendix)


이 튜토리얼에서는 SDF 파일을 다뤘습니다. 하지만 SMILES가 담긴 CSV 파일을 입력으로 사용하면, 데이터프레임의 원자 순서가 원래 원자 순서와 일치하지 않습니다. 각 분자에 대해 원래 원자 순서를 복원하려면(이를 메인 데이터프레임에 담으려면), RDKit의 `Chem.rdmolfiles.CanonicalRankAtoms`를 사용해야 합니다. 아래에 이를 위한 유틸리티들이 있습니다.

(입력 분자에서와 같은) 원자 id 열을 추가하면, 그 결과 데이터프레임을 "python-rdkit-deepchem" 환경 밖의 다른 소프트웨어로 분석하는 데 사용할 수 있습니다.


In [ ]:
def get_mapping(mols, mol_names): 
    """매핑 수행:
    원래 원자 번호 <-> 순위(ranking) 후 원자 번호(위치)
    (둘 다 1부터 시작)"""
    # mols  - RDKit 분자들
    # names - 임의의 문자열 시퀀스
    # 반환: 중첩 리스트 [[분자, [원자, 원자, ..]], [...]]
    assert(len(mols)==len(mol_names))
    mapping = []
    for m,n in zip(mols, mol_names):
        atom_ids = [i+1 for i in list(Chem.rdmolfiles.CanonicalRankAtoms(m))]
        mapping.append([n, atom_ids])
    return mapping


In [ ]:
def append_atomid_col(df, mapping):
    # 올바른 원자 번호(위치) 열을 추가합니다.
    for i in mapping:
        df.loc[i[0],"AtomID"] = i[1]
    return df   


# 참고 문헌(Bibliography)

1. Polishchuk, P., O. Tinkov, T. Khristova, L. Ognichenko, A. Kosinskaya, A. Varnek & V. Kuz'min (2016) Structural and Physico-Chemical Interpretation (SPCI) of QSAR Models and Its Comparison with Matched Molecular Pair Analysis. Journal of Chemical Information and Modeling, 56, 1455-1469.

2. Riniker, S. & G. Landrum (2013) Similarity maps - a visualization strategy for molecular fingerprints and machine-learning methods. Journal of Cheminformatics, 5, 43.

3. M. Matveieva, M. T. D. Cronin, P. Polishchuk, Mol. Inf. 2019, 38, 1800084. 

4. Matveieva, M., Polishchuk, P. Benchmarks for interpretation of QSAR models. J Cheminform 13, 41 (2021). https://doi.org/10.1186/s13321-021-00519-x
